# Uvoz potrebnih biblioteka

In [132]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Učitavanje skupa podataka


In [133]:
from google.colab import files

uploaded = files.upload()

Saving Airbnb_Open_Data.csv to Airbnb_Open_Data (4).csv


In [134]:
df = pd.read_csv(
    "/content/Airbnb_Open_Data.csv",
    engine="python",
    on_bad_lines="skip"
)

# Osnovni pregled skupa podataka

In [135]:
df.shape

(51369, 26)

In [136]:
df.head()

,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,...,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license;;;;;
0,1001254,Clean & quiet apt home by the park,8.001449e+10,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,$193,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,;;;;;
1,1002102,Skylit Midtown Castle,5.233517e+10,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,$28,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,;;;;;
2,1002755,NaN,8.509833e+10,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,$74,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,NaN,;;;;;
3,1004650,BlissArtsSpace!,6.130061e+10,NaN,Alberta,Brooklyn,Bedford-Stuyvesant,40.68688,-73.95596,United States,...,$14,45.0,49.0,10/5/2017,0.40,5.0,1.0,224.0,Please no shoes in the house so bring slippers...,;;;;;
4,1006859,Cute & Cozy Lower East Side 1 bdrm,1.280143e+09,verified,Miranda,Manhattan,Chinatown,40.71344,-73.99037,United States,...,$64,1.0,160.0,6/9/2019,1.33,3.0,4.0,1.0,NaN,;;;;;


In [137]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51369 entries, 0 to 51368
Data columns (total 26 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              51369 non-null  object 
 1   NAME                            51186 non-null  object 
 2   host id                         51322 non-null  float64
 3   host_identity_verified          51181 non-null  object 
 4   host name                       51126 non-null  object 
 5   neighbourhood group             51306 non-null  object 
 6   neighbourhood                   51317 non-null  object 
 7   lat                             51318 non-null  float64
 8   long                            51318 non-null  float64
 9   country                         51016 non-null  object 
 10  country code                    51273 non-null  object 
 11  instant_bookable                51287 non-null  object 
 12  cancellation_policy             

In [138]:
df.columns

Index(['id', 'NAME', 'host id', 'host_identity_verified', 'host name',
       'neighbourhood group', 'neighbourhood', 'lat', 'long', 'country',
       'country code', 'instant_bookable', 'cancellation_policy', 'room type',
       'Construction year', 'price', 'service fee', 'minimum nights',
       'number of reviews', 'last review', 'reviews per month',
       'review rate number', 'calculated host listings count',
       'availability 365', 'house_rules', 'license;;;;;'],
      dtype='object')

# Uređivanje naziva stupaca radi lakšeg rada

In [139]:
df = df.rename(columns={'license;;;;;': 'license'})

In [140]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(';', '', regex=False)
)

# Analiza nedostajućih vrijednosti

In [141]:
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

,0
house_rules,36447
last_review,7566
reviews_per_month,7563
country,353
availability_365,278
minimum_nights,246
host_name,243
review_rate_number,213
price,202
calculated_host_listings_count,198


# Provjera i uklanjanje dupliciranih redaka


In [142]:
df.duplicated().sum()

np.int64(298)

In [143]:
df = df.drop_duplicates()

# Osnovna statistička analiza numeričkih atributa

In [144]:
df.describe()

,host_id,lat,long,construction_year,minimum_nights,number_of_reviews,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365
count,5.104900e+04,51045.000000,51045.00000,50958.000000,50856.000000,50943.000000,43561.000000,50887.000000,50898.000000,50818.000000
mean,4.923543e+10,40.728609,-73.94957,2012.478845,8.492272,28.682115,1.386300,3.308468,8.738654,144.535381
std,2.853310e+10,0.056275,0.05032,5.773513,32.628148,52.111362,1.825957,1.270332,34.134202,135.341265
min,1.236005e+08,40.499790,-74.24984,2003.000000,-365.000000,0.000000,0.010000,1.000000,1.000000,-10.000000
25%,2.460424e+10,40.688930,-73.98294,2007.000000,2.000000,2.000000,0.230000,2.000000,1.000000,5.000000
50%,4.907810e+10,40.722830,-73.95464,2012.000000,3.000000,8.000000,0.770000,3.000000,1.000000,104.000000
75%,7.388642e+10,40.763110,-73.93245,2017.000000,5.000000,31.000000,2.000000,4.000000,3.000000,275.000000
max,9.876313e+10,40.916970,-73.70522,2022.000000,5645.000000,1024.000000,90.000000,5.000000,332.000000,426.000000


# Pretvaranje atributa price iz tekstualnog u numerički oblik

In [145]:
df['price'] = (
    df['price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
)

df['price'] = pd.to_numeric(df['price'], errors='coerce')

# Pretvaranje atributa service_fee iz tekstualnog u numerički oblik

In [146]:
df['service_fee'] = (
    df['service_fee']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
)

df['service_fee'] = pd.to_numeric(df['service_fee'], errors='coerce')

# Uklanjanje atributa koji nisu korisni za daljnju analizu

In [147]:
df = df.drop(columns=['license'])

In [148]:
df = df.drop(columns=['house_rules'])

In [149]:
df = df.drop(columns=['last_review'])

# Imputacija nedostajećih numeričkih vrijednosti medijanom

In [150]:
num_cols = [
    'price',
    'service_fee',
    'minimum_nights',
    'number_of_reviews',
    'reviews_per_month',
    'review_rate_number',
    'calculated_host_listings_count',
    'availability_365'
]

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Imputacija nedostajeućih kategorijskih vrijednosti najčešćom vrijednošću

In [151]:
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

/tmp/ipykernel_990/2165584795.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])


# Provjera nelogičnih vrijednosti u atributima minimum_nights i availability_365

In [152]:
df[df['minimum_nights'] < 0]

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,...,room_type,construction_year,price,service_fee,minimum_nights,number_of_reviews,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365
150,1233854,Charming Nolita Apartment!!,7.389895e+09,verified,Belinda,Manhattan,Nolita,40.72094,-73.99706,United States,...,Entire home/apt,2008.0,874.0,175.0,-10.0,68.0,0.69,5.0,1.0,13.0
157,1244900,Cozy apartment in a brownstone,8.118689e+10,verified,Adelaide,Manhattan,Harlem,40.80497,-73.95016,United States,...,Entire home/apt,2021.0,920.0,184.0,-12.0,203.0,2.14,5.0,3.0,77.0
198,1291294,Chateau Style Brooklyn Loft for Singles or Cou...,2.631537e+09,verified,Carlos,Brooklyn,Bedford-Stuyvesant,40.68967,-73.95445,United States,...,Entire home/apt,2022.0,413.0,83.0,-3.0,42.0,0.44,5.0,1.0,292.0
18244,24474086,2bd BOUTIQUE Apartament in the heart of MANHA...,2.679070e+09,unconfirmed,Tom,Manhattan,Hell's Kitchen,40.76694,-73.98773,United States,...,Entire home/apt,2009.0,711.0,142.0,-365.0,13.0,5.91,4.0,4.0,0.0
18262,24495073,Newly Renovated Garden Apartment,9.846973e+10,verified,Margie,Brooklyn,Bedford-Stuyvesant,40.68470,-73.94350,United States,...,Entire home/apt,2022.0,85.0,17.0,-200.0,3.0,1.06,2.0,1.0,157.0
35708,39523709,Amazing location! 10ft from L train,6.213254e+10,verified,Giorgia & Benjamin,Brooklyn,Williamsburg,40.71534,-73.94906,United States,...,Private room,2012.0,328.0,66.0,-125.0,146.0,1.78,1.0,1.0,46.0


In [153]:
df[df['availability_365'] < 0]

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,...,room_type,construction_year,price,service_fee,minimum_nights,number_of_reviews,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365
54,1078106,FLAT MACDONOUGH,1.235955e+10,unconfirmed,Frederick,Brooklyn,Bedford-Stuyvesant,40.68296,-73.93662,United States,...,Entire home/apt,2010.0,593.0,119.0,3.0,227.0,2.09,5.0,2.0,-10.0
114,1171444,Water View King Bed Hotel Room,9.212481e+10,verified,Thompson,Manhattan,West Village,40.72966,-74.00243,United States,...,Entire home/apt,2010.0,935.0,187.0,30.0,39.0,0.44,2.0,1.0,-9.0
170,1256499,Beautiful Brooklyn Oasis,6.469152e+10,verified,Julia,Brooklyn,Crown Heights,40.67212,-73.95060,United States,...,Entire home/apt,NaN,293.0,59.0,2.0,29.0,0.35,2.0,1.0,-4.0
182,1271963,Gorgeous 1 bdrm in huge duplex!,8.091771e+10,verified,Julian,Manhattan,Harlem,40.80224,-73.94558,United States,...,Private room,2016.0,424.0,85.0,2.0,17.0,0.18,3.0,2.0,-9.0
201,1293503,Columbus Circle Luxury Bldg - Private Room&Bath,8.641726e+10,verified,Derek,Manhattan,Hell's Kitchen,40.77090,-73.99181,United States,...,Private room,2021.0,619.0,124.0,28.0,43.0,0.45,5.0,2.0,-4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44537,48777495,Upscale 1 Bedroom Hell's Kitchen Apartment,5.044231e+10,verified,Kara,Manhattan,Hell's Kitchen,40.76045,-73.99810,United States,...,Entire home/apt,2017.0,471.0,94.0,30.0,1.0,0.04,4.0,121.0,-2.0
44579,48824993,Deluxe 1-Bedroom near Empire State Building!,3.223687e+09,unconfirmed,Kara,Manhattan,Midtown,40.75186,-73.97062,United States,...,Entire home/apt,2008.0,979.0,196.0,30.0,0.0,0.77,2.0,121.0,-8.0
44611,48871939,Private Condo Room w/ Patio,3.536882e+10,verified,Agustina,Bronx,Longwood,40.81937,-73.90978,United States,...,Private room,2005.0,614.0,123.0,2.0,68.0,1.83,2.0,1.0,-7.0
44633,48905077,Comfortable 2 BR in East Village/Cooper Square,6.631924e+10,unconfirmed,Cooper,Manhattan,East Village,40.72627,-73.99145,United States,...,Entire home/apt,2005.0,335.0,67.0,3.0,27.0,0.76,5.0,1.0,-4.0


In [154]:
df[df['availability_365'] > 365]

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,...,room_type,construction_year,price,service_fee,minimum_nights,number_of_reviews,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365
9,1020114,back room/bunk beds,2.506662e+10,verified,Alfred,Manhattan,Harlem,40.82130,-73.95318,United States,...,Private room,2021.0,545.0,109.0,3.0,273.0,2.37,3.0,3.0,411.0
44,1058223,Monkey Retreat Manhattan,6.202860e+10,verified,Kelsey,Manhattan,Washington Heights,40.83927,-73.94281,United States,...,Private room,2007.0,306.0,61.0,3.0,68.0,0.60,3.0,1.0,393.0
46,1062089,Sunny 2-story Brooklyn townhouse w deck and ga...,8.520146e+10,verified,Alen,Brooklyn,Gowanus,40.68157,-73.98989,United States,...,Entire home/apt,2011.0,528.0,106.0,30.0,19.0,0.20,2.0,1.0,400.0
52,1073135,ACCOMMODATIONS GALORE #1,9.052578e+10,unconfirmed,Daniel,Manhattan,Harlem,40.81618,-73.94894,United States,...,Entire home/apt,2005.0,449.0,90.0,3.0,155.0,1.42,3.0,3.0,410.0
55,1078658,Huge Private Floor at The Waverly,6.534697e+09,verified,Adrian,Brooklyn,Clinton Hill,40.68630,-73.96765,United States,...,Private room,2012.0,490.0,98.0,3.0,84.0,0.77,2.0,3.0,378.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44657,48954231,East 63rd street 1bd Serviced Apartment,3.023627e+10,verified,Ken,Manhattan,Upper East Side,40.76371,-73.96153,United States,...,Entire home/apt,2012.0,374.0,75.0,30.0,1.0,0.03,1.0,87.0,394.0
44663,48960307,Spacious Private BR in Chelsea,2.752756e+10,unconfirmed,Ryan,Manhattan,Chelsea,40.74675,-73.99555,United States,...,Private room,2011.0,155.0,31.0,2.0,6.0,0.16,5.0,1.0,415.0
44670,48968039,420 friendly Beautiful rm with plenty of sunlight,1.733888e+10,verified,Sunny,Brooklyn,Bedford-Stuyvesant,40.68285,-73.91484,United States,...,Private room,2022.0,543.0,109.0,14.0,32.0,0.92,4.0,4.0,370.0
44678,48973009,Private Green Apple Brownstone Studio Harlem NYC,6.406383e+10,unconfirmed,Ephraim,Manhattan,Harlem,40.81589,-73.94648,United States,...,Entire home/apt,2003.0,749.0,150.0,4.0,125.0,3.37,4.0,4.0,409.0


# Zamjena nelogičnih vrijednosti s NaN

In [155]:
df.loc[df['availability_365'] < 0, 'availability_365'] = np.nan

In [156]:
df.loc[df['availability_365'] > 365, 'availability_365'] = np.nan

In [157]:
df.loc[df['minimum_nights'] < 0, 'minimum_nights'] = np.nan

# Ponovna imputacija nakon uklanjanja nelogičnih vrijednosti

In [158]:
df['minimum_nights'] = df['minimum_nights'].fillna(
    df['minimum_nights'].median()
)

df['availability_365'] = df['availability_365'].fillna(
    df['availability_365'].median()
)

In [167]:
df["neighbourhood_group"].value_counts()

,count
neighbourhood_group,
Manhattan,22284
Brooklyn,20331
Queens,6485
Bronx,1465
Staten Island,506


In [160]:
df[df["neighbourhood_group"] == "manhatan"]

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,...,room_type,construction_year,price,service_fee,minimum_nights,number_of_reviews,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365
6,1011277,Chelsea Perfect,7.386253e+10,verified,Alberta,manhatan,Chelsea,40.74192,-73.99501,United States,...,Private room,2008.0,460.0,105.0,1.0,260.0,2.12,3.0,1.0,325.0


In [161]:
df["neighbourhood_group"] = df["neighbourhood_group"].replace(
    "manhatan",
    "Manhattan"
)

# Završna provjera očišćenog skupa podataka

In [162]:
df.isnull().sum().sort_values(ascending=False)

,0
construction_year,113
lat,26
long,26
host_id,22
host_identity_verified,0
id,0
name,0
neighbourhood,0
neighbourhood_group,0
host_name,0


In [163]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 51071 entries, 0 to 51095
Data columns (total 23 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              51071 non-null  object 
 1   name                            51071 non-null  object 
 2   host_id                         51049 non-null  float64
 3   host_identity_verified          51071 non-null  object 
 4   host_name                       51071 non-null  object 
 5   neighbourhood_group             51071 non-null  object 
 6   neighbourhood                   51071 non-null  object 
 7   lat                             51045 non-null  float64
 8   long                            51045 non-null  float64
 9   country                         51071 non-null  object 
 10  country_code                    51071 non-null  object 
 11  instant_bookable                51071 non-null  bool   
 12  cancellation_policy             51071

In [164]:
df.describe()

,host_id,lat,long,construction_year,price,service_fee,minimum_nights,number_of_reviews,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365
count,5.104900e+04,51045.000000,51045.00000,50958.000000,51071.000000,51071.000000,51071.000000,51071.000000,51071.000000,51071.000000,51071.000000,51071.000000
mean,4.923543e+10,40.728609,-73.94957,2012.478845,525.931762,105.221065,8.483503,28.630279,1.295673,3.307356,8.712440,137.790135
std,2.853310e+10,0.056275,0.05032,5.773513,273.955826,54.825779,32.500701,52.056288,1.700432,1.268176,34.079304,129.216702
min,1.236005e+08,40.499790,-74.24984,2003.000000,50.000000,10.000000,1.000000,0.000000,0.010000,1.000000,1.000000,0.000000
25%,2.460424e+10,40.688930,-73.98294,2007.000000,288.000000,58.000000,2.000000,2.000000,0.290000,2.000000,1.000000,6.000000
50%,4.907810e+10,40.722830,-73.95464,2012.000000,527.000000,105.000000,3.000000,8.000000,0.770000,3.000000,1.000000,98.000000
75%,7.388642e+10,40.763110,-73.93245,2017.000000,762.000000,152.000000,5.000000,31.000000,1.740000,4.000000,3.000000,258.000000
max,9.876313e+10,40.916970,-73.70522,2022.000000,999.000000,239.000000,5645.000000,1024.000000,90.000000,5.000000,332.000000,365.000000


# Spremanje očišćenog skupa podataka

In [168]:
df.to_csv(
    "airbnb_clean.csv",
    index=False
)

In [169]:
from google.colab import files

files.download("airbnb_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>